In [1]:
import os
import time
import random
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

from skimage.metrics import (
    peak_signal_noise_ratio,
    structural_similarity,
    mean_squared_error
)


print("DA-NAF Lite Residual U-Net Restoration")
print("PyTorch:", torch.__version__)

DA-NAF Lite Residual U-Net Restoration
PyTorch: 2.5.1+cu121


In [2]:
torch.cuda.empty_cache()

print("Model recreated")

Model recreated


In [3]:
SEED = 42


random.seed(SEED)
np.random.seed(SEED)

torch.manual_seed(SEED)


if torch.cuda.is_available():

    torch.cuda.manual_seed_all(SEED)



device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


print(
    "Device:",
    device
)


if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )


scaler = torch.cuda.amp.GradScaler()

Device: cuda
GPU: Quadro GV100


/tmp/ipykernel_201023/1342682282.py:37: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


In [4]:
# Cell 3 — Dataset Paths

DATA_DIR = "../data"


GT_PATH = os.path.join(
    DATA_DIR,
    "train",
    "GT"
)


NOISY_PATH = os.path.join(
    DATA_DIR,
    "train",
    "NoisyLR"
)


TEST_PATH = os.path.join(
    DATA_DIR,
    "test",
    "NoisyLR"
)



MODEL_DIR = "../models/da_naf_lite_resunet"

RESULT_DIR = "../results/da_naf_lite_resunet"



os.makedirs(
    MODEL_DIR,
    exist_ok=True
)


os.makedirs(
    RESULT_DIR,
    exist_ok=True
)



BEST_MODEL_PATH = os.path.join(
    MODEL_DIR,
    "da_naf_lite_best.pth"
)



print("Paths ready")

print(
    "GT:",
    GT_PATH
)

print(
    "NOISY:",
    NOISY_PATH
)

Paths ready
GT: ../data/train/GT
NOISY: ../data/train/NoisyLR


In [5]:
# Cell 4 — Dataset

class KLADataset(Dataset):

    def __init__(
        self,
        noisy_dir,
        gt_dir,
        files
    ):

        self.noisy_dir = noisy_dir
        self.gt_dir = gt_dir

        self.files = files



    def __len__(self):

        return len(
            self.files
        )



    def __getitem__(
        self,
        idx
    ):


        filename = self.files[idx]


        noisy = np.load(
            os.path.join(
                self.noisy_dir,
                filename
            )
        ).astype(
            np.float32
        )


        gt = np.load(
            os.path.join(
                self.gt_dir,
                filename
            )
        ).astype(
            np.float32
        )


        noisy = torch.from_numpy(
            noisy
        ).unsqueeze(0)


        gt = torch.from_numpy(
            gt
        ).unsqueeze(0)



        return noisy, gt

In [6]:
# Cell 5 — Split Dataset


files = sorted(
    [
        f
        for f in os.listdir(GT_PATH)
        if f.endswith(".npy")
    ]
)


print(
    "Total pairs:",
    len(files)
)



np.random.seed(SEED)


indices = np.random.permutation(
    len(files)
)



train_end = int(
    0.80 * len(files)
)


val_end = int(
    0.90 * len(files)
)



train_files = [
    files[i]
    for i in indices[:train_end]
]


val_files = [
    files[i]
    for i in indices[
        train_end:val_end
    ]
]


test_files = [
    files[i]
    for i in indices[
        val_end:
    ]
]



print(
    "Train:",
    len(train_files)
)


print(
    "Validation:",
    len(val_files)
)


print(
    "Test:",
    len(test_files)
)

Total pairs: 3200
Train: 2560
Validation: 320
Test: 320


In [7]:
# Cell 6 — DataLoaders


BATCH_SIZE = 4



train_dataset = KLADataset(
    NOISY_PATH,
    GT_PATH,
    train_files
)



val_dataset = KLADataset(
    NOISY_PATH,
    GT_PATH,
    val_files
)



test_dataset = KLADataset(
    NOISY_PATH,
    GT_PATH,
    test_files
)



train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)



val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)



test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)



print(
    "DataLoaders ready"
)

DataLoaders ready


In [8]:
class DegradationMap(nn.Module):

    def __init__(self):

        super().__init__()



    def forward(
        self,
        x
    ):

        # local mean

        mean = F.avg_pool2d(
            x,
            kernel_size=7,
            stride=1,
            padding=3
        )


        # local variance

        variance = F.avg_pool2d(
            (x-mean)**2,
            kernel_size=7,
            stride=1,
            padding=3
        )


        degradation = torch.sqrt(
            variance + 1e-8
        )


        return degradation

In [9]:
class NAFBlock(nn.Module):

    def __init__(
        self,
        channels
    ):

        super().__init__()


        self.norm = nn.GroupNorm(
            1,
            channels
        )


        self.conv1 = nn.Conv2d(
            channels,
            channels,
            1
        )


        self.dwconv = nn.Conv2d(
            channels,
            channels,
            3,
            padding=1,
            groups=channels
        )


        self.conv2 = nn.Conv2d(
            channels,
            channels,
            1
        )


        self.beta = nn.Parameter(
            torch.zeros(1)
        )



    def forward(
        self,
        x
    ):

        residual=x


        y=self.norm(x)

        y=self.conv1(y)

        y=self.dwconv(y)

        y=self.conv2(y)



        return residual + self.beta*y

In [10]:
class EncoderBlock(nn.Module):

    def __init__(
        self,
        in_c,
        out_c
    ):

        super().__init__()


        self.block = nn.Sequential(

            nn.Conv2d(
                in_c,
                out_c,
                3,
                padding=1
            ),

            NAFBlock(
                out_c
            ),

            NAFBlock(
                out_c
            )

        )



    def forward(
        self,
        x
    ):

        return self.block(x)

In [11]:
class DecoderBlock(nn.Module):

    def __init__(
        self,
        in_c,
        skip_c,
        out_c
    ):

        super().__init__()


        self.up = nn.ConvTranspose2d(
            in_c,
            out_c,
            kernel_size=2,
            stride=2
        )


        self.fuse = nn.Conv2d(
            out_c + skip_c,
            out_c,
            kernel_size=1
        )


        self.block = nn.Sequential(

            NAFBlock(
                out_c
            ),

            NAFBlock(
                out_c
            )

        )


    def forward(
        self,
        x,
        skip
    ):


        x = self.up(
            x
        )


        # safety resize
        if x.shape[-2:] != skip.shape[-2:]:

            x = F.interpolate(
                x,
                size=skip.shape[-2:],
                mode="bilinear",
                align_corners=False
            )


        x = torch.cat(
            [
                x,
                skip
            ],
            dim=1
        )


        x = self.fuse(
            x
        )


        return self.block(
            x
        )

In [12]:
# Cell 11 — Corrected DA NAF Lite ResUNet


class DANAFLiteResUNet(nn.Module):

    def __init__(
        self,
        channels=32
    ):

        super().__init__()


        self.degradation = DegradationMap()



        self.input_conv = nn.Conv2d(
            2,
            channels,
            kernel_size=3,
            padding=1
        )


        # Encoder

        self.enc1 = EncoderBlock(
            channels,
            channels
        )


        self.down1 = nn.Conv2d(
            channels,
            channels*2,
            kernel_size=4,
            stride=2,
            padding=1
        )


        self.enc2 = EncoderBlock(
            channels*2,
            channels*2
        )


        self.down2 = nn.Conv2d(
            channels*2,
            channels*4,
            kernel_size=4,
            stride=2,
            padding=1
        )



        # Bottleneck

        self.middle = nn.Sequential(

            NAFBlock(
                channels*4
            ),

            NAFBlock(
                channels*4
            )
        )



        # Decoder (Corrected)

        self.decoder2 = DecoderBlock(
            channels*4,
            channels*2,
            channels*2
        )


        self.decoder1 = DecoderBlock(
            channels*2,
            channels,
            channels
        )



        # Output

        self.output = nn.Conv2d(
            channels,
            1,
            kernel_size=3,
            padding=1
        )




    def forward(
        self,
        x
    ):


        original_input = x



        # degradation map

        deg = self.degradation(
            x
        )


        x = torch.cat(
            [
                x,
                deg
            ],
            dim=1
        )


        x = self.input_conv(
            x
        )



        # Encoder

        e1 = self.enc1(
            x
        )


        x = self.down1(
            e1
        )


        e2 = self.enc2(
            x
        )


        x = self.down2(
            e2
        )



        # Bottleneck

        x = self.middle(
            x
        )



        # Decoder

        x = self.decoder2(
            x,
            e2
        )


        x = self.decoder1(
            x,
            e1
        )



        # Output

        x = self.output(x)
        
        
        x = F.interpolate(
            x,
            scale_factor=2,
            mode="bilinear",
            align_corners=False
        )
        
        
        base = F.interpolate(
            original_input,
            scale_factor=2,
            mode="bilinear",
            align_corners=False
        )
        
        
        x = base + x
        
        
        return torch.clamp(
            x,
            0,
            1
)

In [13]:
# Cell 12 — Model Initialization


model = DANAFLiteResUNet(
    channels=32
).to(device)



total_params = sum(
    p.numel()
    for p in model.parameters()
)



trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)



print(
    "Total parameters:",
    total_params/1e6,
    "M"
)


print(
    "Trainable parameters:",
    trainable_params/1e6,
    "M"
)

Total parameters: 0.377963 M
Trainable parameters: 0.377963 M


In [14]:
# Cell 13A — Check output size


noisy, gt = next(
    iter(train_loader)
)


noisy = noisy.to(device)


with torch.no_grad():

    pred = model(
        noisy
    )


print(
    "Noisy:",
    noisy.shape
)


print(
    "Prediction:",
    pred.shape
)


print(
    "GT:",
    gt.shape
)

Noisy: torch.Size([4, 1, 128, 128])
Prediction: torch.Size([4, 1, 256, 256])
GT: torch.Size([4, 1, 256, 256])


In [15]:
model = DANAFLiteResUNet(
    channels=32
).to(device)


dummy = torch.randn(
    1,
    1,
    128,
    128
).to(device)


out = model(dummy)


print(out.shape)

torch.Size([1, 1, 256, 256])


In [16]:
# Cell 13 — Differentiable SSIM


def gaussian_window(
    size=11,
    sigma=1.5,
    channels=1
):

    coords = torch.arange(
        size
    ).float()


    coords -= size//2


    g = torch.exp(
        -(coords**2) /
        (2*sigma*sigma)
    )


    g /= g.sum()


    window = (
        g[:,None]
        *
        g[None,:]
    )


    return window.expand(
        channels,
        1,
        size,
        size
    )



def differentiable_ssim(
    img1,
    img2
):

    channels = img1.shape[1]


    window = gaussian_window(
        channels=channels
    ).to(
        img1.device
    )


    mu1 = F.conv2d(
        img1,
        window,
        padding=5,
        groups=channels
    )


    mu2 = F.conv2d(
        img2,
        window,
        padding=5,
        groups=channels
    )


    sigma1 = F.conv2d(
        img1*img1,
        window,
        padding=5,
        groups=channels
    ) - mu1**2


    sigma2 = F.conv2d(
        img2*img2,
        window,
        padding=5,
        groups=channels
    ) - mu2**2


    sigma12 = F.conv2d(
        img1*img2,
        window,
        padding=5,
        groups=channels
    ) - mu1*mu2


    C1 = 0.01**2
    C2 = 0.03**2


    ssim = (
        (2*mu1*mu2+C1)
        *
        (2*sigma12+C2)
    ) / (
        (mu1**2+mu2**2+C1)
        *
        (sigma1+sigma2+C2)
        +
        1e-8
    )


    return ssim.mean()

In [17]:
# Cell 14 — Restoration Loss


def restoration_loss(
    pred,
    gt
):


    l1 = F.l1_loss(
        pred,
        gt
    )


    ssim = differentiable_ssim(
        pred,
        gt
    )


    loss = (
        l1
        +
        0.1*(1-ssim)
    )


    return loss

In [18]:
# Cell 15


EPOCHS = 40


optimizer = optim.AdamW(
    model.parameters(),
    lr=2e-4,
    weight_decay=1e-4
)


scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=EPOCHS
)



print("Optimizer ready")

Optimizer ready


In [19]:
# Cell 16


def validate(
    model,
    loader
):

    model.eval()


    psnr_list=[]
    ssim_list=[]


    with torch.no_grad():

        for noisy,gt in loader:


            noisy = noisy.to(device)

            gt = gt.to(device)



            pred = model(
                noisy
            )


            pred_np = (
                pred.cpu()
                .numpy()
            )


            gt_np = (
                gt.cpu()
                .numpy()
            )


            psnr_list.append(
                peak_signal_noise_ratio(
                    gt_np[0,0],
                    pred_np[0,0],
                    data_range=1
                )
            )


            ssim_list.append(
                structural_similarity(
                    gt_np[0,0],
                    pred_np[0,0],
                    data_range=1
                )
            )


    return (
        np.mean(psnr_list),
        np.mean(ssim_list)
    )

In [22]:
# Cell 17

from tqdm import tqdm


EPOCHS = 40


best_psnr = 0



for epoch in range(EPOCHS):


    model.train()


    running_loss = 0



    loop = tqdm(
        train_loader,
        desc=f"Epoch {epoch+1}/{EPOCHS}"
    )


    for noisy, gt in loop:


        noisy = noisy.to(
            device,
            non_blocking=True
        )

        gt = gt.to(
            device,
            non_blocking=True
        )


        optimizer.zero_grad()



        with torch.cuda.amp.autocast():


            pred = model(
                noisy
            )


            loss = restoration_loss(
                pred,
                gt
            )



        scaler.scale(
            loss
        ).backward()



        scaler.step(
            optimizer
        )


        scaler.update()



        running_loss += loss.item()



        loop.set_postfix(
            loss=loss.item()
        )



    scheduler.step()



    val_psnr, val_ssim = validate(
        model,
        val_loader
    )


    avg_loss = (
        running_loss /
        len(train_loader)
    )



    print(
        "\nEpoch:",
        epoch+1
    )

    print(
        "Loss:",
        avg_loss
    )

    print(
        "PSNR:",
        val_psnr
    )

    print(
        "SSIM:",
        val_ssim
    )



    if val_psnr > best_psnr:


        best_psnr = val_psnr


        torch.save(
            {
                "model_state":
                    model.state_dict(),

                "psnr":
                    val_psnr,

                "ssim":
                    val_ssim
            },

            "../models/da_naf_lite_resunet_best.pth"
        )


        print(
            "⭐ Best model saved"
        )

Epoch 1/40:   0%|          | 0/640 [00:00<?, ?it/s]/tmp/ipykernel_201023/736090737.py:47: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/40:  29%|██▉       | 187/640 [00:08<00:19, 22.75it/s, loss=0.0977]


KeyboardInterrupt: 

In [23]:
# Cell 18 — Load Best Model


BEST_MODEL_PATH = "../models/da_naf_lite_resunet_best.pth"


checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=device
)


model.load_state_dict(
    checkpoint["model_state"]
)


model.to(device)

model.eval()


print(
    "Best model loaded"
)


print(
    "Best PSNR:",
    checkpoint["psnr"]
)


print(
    "Best SSIM:",
    checkpoint["ssim"]
)

Best model loaded
Best PSNR: 25.524523655766615
Best SSIM: 0.6284304984877347


/tmp/ipykernel_201023/505145249.py:7: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(


In [25]:
# Cell — Final Shape Check

model.eval()

with torch.no_grad():

    noisy, gt = test_dataset[0]

    noisy = noisy.unsqueeze(0).to(device)

    pred = model(noisy)


print("Input :", noisy.shape)
print("Output:", pred.shape)
print("GT    :", gt.unsqueeze(0).shape)

Input : torch.Size([1, 1, 128, 128])
Output: torch.Size([1, 1, 256, 256])
GT    : torch.Size([1, 1, 256, 256])


In [28]:
# Cell — Final Evaluation

psnr_scores=[]
ssim_scores=[]
mse_scores=[]

model.eval()

with torch.no_grad():

    for i in range(len(test_dataset)):

        noisy, gt = test_dataset[i]


        noisy = noisy.unsqueeze(0).to(device)
        gt = gt.numpy()


        pred = model(noisy)


        pred = pred.squeeze().cpu().numpy()


        pred = np.clip(
            pred,
            0,
            1
        )


        psnr_scores.append(
            peak_signal_noise_ratio(
                gt,
                pred,
                data_range=1
            )
        )


        ssim_scores.append(
            structural_similarity(
                gt,
                pred,
                data_range=1
            )
        )


        mse_scores.append(
            mean_squared_error(
                gt,
                pred
            )
        )


print("FINAL RESULTS")
print("----------------")
print("PSNR :", np.mean(psnr_scores))
print("SSIM :", np.mean(ssim_scores))
print("MSE  :", np.mean(mse_scores))

ValueError: Input images must have the same dimensions.